In [ ]:
!pip install openai

In [ ]:
import json
import random
import time
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

In [ ]:
# 경로 설정
from google.colab import files
uploaded = files.upload()  # 파일 선택 창 뜸

SEED_PATH = "contract_risk_seed_v1.json"
OUTPUT_PATH = "contract_risk_augmented_v1.json"
LOG_PATH = "filter_log.txt"
MODEL = "gpt-4o"
TARGET_PER_VERDICT = 6  # 유형×판정당 생성 목표 수

Saving contract_risk_seed_v1.json to contract_risk_seed_v1 (1).json


In [ ]:
TYPE_CONFIG = {
    "지체상금 상한 미설정": {
        "ref": (
            "지방자치단체 용역계약 일반조건 제7절 제1항 가(행안부 예규 제324호): "
            "계약서에 정한 지연배상금율에 계약금액을 곱하여 산출한 금액을 납부하여야 한다.\n"
            "지방계약법 시행령 제90조③: 지연배상금 총액은 계약금액의 100분의 30을 초과할 수 없다."
        ),
        "upper_law": "지방계약법 시행령 제90조③",
        "key_value": "지체상금 상한 30%",
        "정상_hint": "지체상금 상한 30% 명시",
        "누락_hint": "지체상금 관련 조항 자체 없음 또는 상한 미기재",
        "위반_hint": "상한을 30% 미만으로 명시",
        "양방향_위반": False,
    },
    "지체상금률 과다 설정": {
        "ref": (
            "지방자치단체 용역계약 일반조건 제7절 제1항 가(행안부 예규 제324호): "
            "계약서에 정한 지연배상금율에 계약금액을 곱하여 산출한 금액을 납부하여야 한다.\n"
            "지방계약법 시행규칙 제75조③: 용역의 지연배상금률은 1천분의 1.3이다."
        ),
        "upper_law": "지방계약법 시행규칙 제75조③",
        "key_value": "용역 지연배상금률 1천분의 1.3 (0.13%/일)",
        "정상_hint": "요율 1천분의 1.3 명시",
        "누락_hint": "요율 미기재 또는 모호하게 기재",
        "위반_hint": "요율을 1천분의 1.3 초과로 명시",
        "양방향_위반": False,
    },
    "대금 지급 기한 미설정": {
        "ref": (
            "지방계약법 시행령 제67조①: "
            "청구를 받은 날부터 5일 이내에 지급하여야 한다."
        ),
        "upper_law": "지방계약법 제18조②",
        "key_value": "청구일로부터 5일 이내 지급",
        "정상_hint": "청구일로부터 5일 이내 명시",
        "누락_hint": "지급 기한 자체 없음",
        "위반_hint": "지급 기한을 5일 초과로 명시",
        "양방향_위반": False,
    },
    "합의 없는 일방적 과업 추가": {
        "ref": (
            "지방자치단체 용역계약 일반조건 제6절 제1항 나(행안부 예규 제324호): "
            "과업내용의 변경은 그 변경이 필요한 부분의 이행 전에 완료하여야 하며, "
            "긴급한 경우 계약상대자와 협의하여 변경시기를 명확히 정하여야 한다."
        ),
        "upper_law": "소프트웨어 진흥법 제50조③",
        "key_value": "과업 추가 시 사전 협의 의무",
        "정상_hint": "사전 협의 의무 명시",
        "누락_hint": "과업추가 절차 없거나 협의 의무 누락",
        "위반_hint": "갑의 일방적 지시만 명시, 협의 절차 없음",
        "양방향_위반": False,
    },
    "추가 과업 비용 미지급": {
        "ref": (
            "지방자치단체 용역계약 일반조건 제6절 제1항 라·마(행안부 예규 제324호): "
            "과업내용의 변경을 지시하거나 승인한 경우 계약금액 조정은 시행령 제74조를 준용하며, "
            "발주기관은 청구를 받은 날로부터 30일 이내에 계약금액을 조정하여야 한다."
        ),
        "upper_law": "지방계약법 시행령 제74조",
        "key_value": "추가 과업 대가 지급, 청구일로부터 30일 이내 조정",
        "정상_hint": "추가 과업 대가 지급 의무 명시",
        "누락_hint": "추가 과업 발생 시 대가 기준 없음",
        "위반_hint": "추가 과업 무상 수행 강제",
        "양방향_위반": False,
    },
    "계약금액 조정 없는 과업변경": {
        "ref": (
            "지방자치단체 용역계약 일반조건 제6절 제1항 라(행안부 예규 제324호): "
            "과업내용의 변경을 지시하거나 승인한 경우 계약금액 조정은 "
            "지방계약법 시행령 제74조제1항 내지 제6항의 계약금액 조정 규정을 준용한다."
        ),
        "upper_law": "지방계약법 시행령 제74조",
        "key_value": "과업변경 시 계약금액 조정 의무 (시행령 제74조 준용)",
        "정상_hint": "과업변경 시 계약금액 조정 의무 명시",
        "누락_hint": "과업변경 가능하나 금액 조정 기준 없음",
        "위반_hint": "과업변경에도 계약금액 불변 명시",
        "양방향_위반": False,
    },
    "갑의 일방적 해지권": {
        "ref": (
            "지방자치단체 용역계약 일반조건 제7절 제4항 다(행안부 예규 제324호): "
            "발주기관은 계약을 해제 또는 해지할 경우 해제 또는 해지한 날부터 14일 이내에 "
            "기수행 부분에 대한 미지급 대가 및 투입된 인력·자재·장비의 철수 비용을 "
            "계약상대자에게 지급하여야 한다."
        ),
        "upper_law": None,
        "key_value": "해지 후 14일 이내 기수행 대가 + 철수비용 지급",
        "정상_hint": "해지 시 14일 이내 대가 + 철수비용 + 보증금 반환 명시",
        "누락_hint": "해지권은 있으나 대가 지급 또는 철수비용 규정 누락",
        "위반_hint": "철수비용 청구 명시적 금지",
        "양방향_위반": False,
    },
    "을의 해제권 배제/제한": {
        "ref": (
            "지방자치단체 용역계약 일반조건 제7절 제5항 가(행안부 예규 제324호): "
            "계약상대자는 계약내용의 변경으로 계약금액이 100분의 40 이상 감소되었을 때 "
            "또는 용역수행 정지기간이 계약기간의 100분의 50을 초과하였을 경우에는 "
            "해당 계약을 해제 또는 해지할 수 있다."
        ),
        "upper_law": None,
        "key_value": "계약금액 40% 이상 감소 또는 정지기간 50% 초과 시 을의 해지권",
        "정상_hint": "해지권 + 해지 시 대가·철수비용 정산 조항 명시",
        "누락_hint": "해지권은 있으나 사후 정산(대가·철수비용) 조항 누락",
        "위반_hint": "을의 해지권 원천 배제",
        "양방향_위반": False,
    },
    "손해배상 범위 일방적 제한": {
        "ref": (
            "지방자치단체 용역계약 일반조건 제8절 제7항 가(행안부 예규 제324호): "
            "계약상대자는 계약의 수행 중 용역목적물 및 제3자에 대한 손해를 부담하여야 한다. "
            "다만, 계약상대자의 책임없는 사유로 인하여 발생한 경우에는 발주기관의 부담으로 한다."
        ),
        "upper_law": None,
        "key_value": "을 원칙 부담 + 을 귀책 없는 경우 갑 부담",
        "정상_hint": "을 원칙 부담 + 을 귀책 없는 경우 갑 부담 명시",
        "누락_hint": "을 부담만 있고 갑 부담(면책) 조항 없음",
        "위반_hint": "귀책 무관 을 전면 부담 또는 을 전면 면책",
        "양방향_위반": True,
    },
}

SYSTEM_PROMPT = """당신은 공공기관 SW 용역계약 조항 분석 전문가입니다.
각 유형별 참고기준에 명시된 법령 조문을 기준으로 계약 조항을 판정하세요.
참고기준 외의 법령을 임의로 적용하지 마세요.

## 판정 기준
- 정상: 참고기준의 핵심 요소(수치, 기한, 절차, 주체)가 전부 명시된 경우
- 누락: 다음 두 가지 중 하나
    1. 관련 조항 자체가 없는 경우 (예: 지체상금 언급 없음, 과업 추가 관련 언급 없음)
    2. 형식은 있으나 핵심 요소가 빠진 경우 (예: 해지권은 있는데 사후 정산 조항 없음)
- 위반: 기준과 다른 값을 명시적으로 써놓은 경우
    수치가 기준 초과/미달이거나, 권리를 명시적으로 박탈하거나, 의무를 명시적으로 면제하는 경우

## 규칙
1. 참고기준 조문 번호와 수치는 절대 변경하지 않는다
2. 판정은 지시에 따라 고정한다
3. 실제 공공기관(교육청·지자체) 계약서 문체로 작성한다
4. 근거 문장은 설명형으로 작성한다 ("~위반입니다", "~불공정 조항입니다" 금지)
   올바른 예: "지방자치단체 용역계약 일반조건(행안부 예규)의 기준에서 정한 요건을 충족하지 않습니다"
5. 수정 방향, 수정 권고, 법적 조언을 절대 포함하지 않는다
   ("~로 수정하시기 바랍니다", "~조항을 추가하세요" 등 금지)
6. JSON 배열만 반환하고 다른 텍스트는 출력하지 않는다"""

print("설정 완료")

설정 완료


In [ ]:
# 함수 정의
BANNED_PHRASES = [
    "수정하시기 바랍니다", "추가하세요", "변경하세요",
    "권고", "법적 조언", "검토하시기", "명시하시기",
    "전문가 상담", "법률 서비스",
]

def auto_filter(item, type_name, verdict, config):
    output = item.get("output", "")
    input_text = item.get("input", "")

    for phrase in BANNED_PHRASES:
        if phrase in output:
            return False, f"banned phrase: {phrase}"

    if not output.startswith(f"판정: {verdict}"):
        return False, f"판정 불일치: {output[:20]}"

    if f"유형: {type_name}" not in output:
        return False, "유형 불일치"

    upper_law = config["upper_law"]
    if upper_law:
        # 상위법령이 있는 경우 output에 명시되어야 함
        if f"관련 상위법령: {upper_law}" not in output:
            return False, "상위법령 표기 누락"
    else:
        # 상위법령이 없는 경우 output에 관련 상위법령 구문이 없어야 함
        if "관련 상위법령:" in output:
            return False, "상위법령 없음인데 표기됨"

    if config["ref"][:20] not in input_text:
        return False, "참고기준 누락"

    return True, "ok"

def make_user_prompt(type_name, verdict, config, examples, n):
    verdict_guide = {
        "정상": (
            "참고기준의 핵심 요소(수치, 기한, 절차, 주체)가 모두 반영된 조항.\n"
            "주어(갑은/계약담당공무원은/발주기관은), 조건절, 문장구조를 기존 예시와 다르게 변형하세요."
        ),
        "누락": (
            "쉬운 누락(핵심 요소 통째로 없음) 또는 어려운 누락(형식은 있으나 핵심 수치·조건 하나만 빠짐).\n"
            "두 가지를 골고루 생성하세요."
        ),
        "위반": (
            "갑 유리형(을의 권리 박탈·의무 가중) 또는 을 유리형(을의 의무 면제·권리 과다 확대).\n"
            f"{'두 방향을 골고루 생성하세요.' if config['양방향_위반'] else '갑 유리형 위반을 생성하세요.'}"
        ),
    }

    example_str = "\n".join([
        f"- 계약조항: {ex['input'].split(chr(10))[1]}\n  output: {ex['output']}"
        for ex in examples
    ])

    upper_law_line = f"\n[관련 상위법령] {config['upper_law']}" if config['upper_law'] else ""
    upper_law_suffix = f" (관련 상위법령: {config['upper_law']})" if config['upper_law'] else ""

    return f"""다음 조건으로 계약 조항 샘플을 {n}개 생성하세요.

[유형] {type_name}
[판정] {verdict}
[판정 지침] {verdict_guide[verdict]}
[참고기준] {config['ref']}
[핵심기준값] {config['key_value']}{upper_law_line}

[기존 예시 - 참고용, 중복 금지]
{example_str}

[반환 형식] 순수 JSON 배열만 반환
[
  {{
    "instruction": "다음 계약 조항의 위험 여부를 판단하세요.",
    "input": "[계약조항]\\n조항 내용\\n\\n[참고기준]\\n{config['ref']}",
    "output": "판정: {verdict}\\n유형: {type_name}\\n근거: ...{upper_law_suffix}"
  }}
]"""

In [ ]:
# 증강 실행
with open(SEED_PATH, encoding="utf-8") as f:
    seed_data = json.load(f)

seed_map = defaultdict(list)
for item in seed_data:
    lines     = item["output"].split("\n")
    verdict   = lines[0].replace("판정: ", "").strip()
    type_name = lines[1].replace("유형: ", "").strip()
    seed_map[(type_name, verdict)].append(item)

results    = list(seed_data)
augmented  = []
filter_log = []

for type_name, config in TYPE_CONFIG.items():
    for verdict in ["정상", "누락", "위반"]:
        existing     = seed_map.get((type_name, verdict), [])
        generated    = []
        attempts     = 0
        max_attempts = TARGET_PER_VERDICT * 3

        while len(generated) < TARGET_PER_VERDICT and attempts < max_attempts:
            batch  = min(3, TARGET_PER_VERDICT - len(generated))
            print(f"[GEN] {type_name} / {verdict} - {batch}개 생성 중... ({len(generated)}/{TARGET_PER_VERDICT})")

            prompt = make_user_prompt(
                type_name, verdict, config,
                examples=existing + generated,
                n=batch
            )

            try:
                resp = client.chat.completions.create(
                    model="gpt-4o",
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": prompt}
                    ],
                    temperature=0.8,
                )
                raw = resp.choices[0].message.content.strip()
                raw = raw.replace("```json", "").replace("```", "").strip()
                batch_result = json.loads(raw)

                for item in batch_result:
                    ok, reason = auto_filter(item, type_name, verdict, config)
                    if ok:
                        generated.append(item)
                    else:
                        filter_log.append(f"[FILTER] {type_name}/{verdict} - {reason}")
                        print(f"  필터 탈락: {reason}")

                attempts += batch
                time.sleep(1)

            except Exception as e:
                print(f"  오류: {e}")
                attempts += batch

        augmented.extend(generated)
        print(f"  -> {len(generated)}개 확보 완료\n")

results.extend(augmented)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

if filter_log:
    with open(LOG_PATH, "w", encoding="utf-8") as f:
        f.write("\n".join(filter_log))

print(f"\n{'='*50}")
print(f"총 {len(results)}개 저장 완료")
print(f"  시드: {len(seed_data)}개")
print(f"  증강: {len(augmented)}개")
print(f"  필터 탈락: {len(filter_log)}개")
print(f"{'='*50}")

[GEN] 지체상금 상한 미설정 / 정상 - 3개 생성 중... (0/6)
[GEN] 지체상금 상한 미설정 / 정상 - 3개 생성 중... (3/6)
  -> 6개 확보 완료

[GEN] 지체상금 상한 미설정 / 누락 - 3개 생성 중... (0/6)
[GEN] 지체상금 상한 미설정 / 누락 - 3개 생성 중... (3/6)
  -> 6개 확보 완료

[GEN] 지체상금 상한 미설정 / 위반 - 3개 생성 중... (0/6)
[GEN] 지체상금 상한 미설정 / 위반 - 3개 생성 중... (3/6)
  -> 6개 확보 완료

[GEN] 지체상금률 과다 설정 / 정상 - 3개 생성 중... (0/6)
[GEN] 지체상금률 과다 설정 / 정상 - 3개 생성 중... (3/6)
  -> 6개 확보 완료

[GEN] 지체상금률 과다 설정 / 누락 - 3개 생성 중... (0/6)
[GEN] 지체상금률 과다 설정 / 누락 - 3개 생성 중... (3/6)
  -> 6개 확보 완료

[GEN] 지체상금률 과다 설정 / 위반 - 3개 생성 중... (0/6)
[GEN] 지체상금률 과다 설정 / 위반 - 3개 생성 중... (3/6)
  -> 6개 확보 완료

[GEN] 대금 지급 기한 미설정 / 정상 - 3개 생성 중... (0/6)
[GEN] 대금 지급 기한 미설정 / 정상 - 3개 생성 중... (3/6)
  -> 6개 확보 완료

[GEN] 대금 지급 기한 미설정 / 누락 - 3개 생성 중... (0/6)
[GEN] 대금 지급 기한 미설정 / 누락 - 3개 생성 중... (3/6)
  -> 6개 확보 완료

[GEN] 대금 지급 기한 미설정 / 위반 - 3개 생성 중... (0/6)
[GEN] 대금 지급 기한 미설정 / 위반 - 3개 생성 중... (3/6)
  -> 6개 확보 완료

[GEN] 합의 없는 일방적 과업 추가 / 정상 - 3개 생성 중... (0/6)
[GEN] 합의 없는 일방적 과업 추가 / 정상 - 3개 생성 중... (3/6)
  -> 6개 확보 

In [ ]:
# 결과 통계 확인

type_verdict = Counter()
for item in results:
    lines   = item["output"].split("\n")
    verdict = lines[0].replace("판정: ", "").strip()
    tname   = lines[1].replace("유형: ", "").strip()
    type_verdict[(tname, verdict)] += 1

print("[유형별 판정 분포]")
for type_name in TYPE_CONFIG:
    counts = [type_verdict.get((type_name, v), 0) for v in ["정상", "누락", "위반"]]
    print(f"  {type_name}: 정상 {counts[0]} / 누락 {counts[1]} / 위반 {counts[2]}")

[유형별 판정 분포]
  지체상금 상한 미설정: 정상 7 / 누락 7 / 위반 7
  지체상금률 과다 설정: 정상 8 / 누락 7 / 위반 7
  대금 지급 기한 미설정: 정상 7 / 누락 7 / 위반 8
  합의 없는 일방적 과업 추가: 정상 7 / 누락 8 / 위반 7
  추가 과업 비용 미지급: 정상 7 / 누락 8 / 위반 7
  계약금액 조정 없는 과업변경: 정상 7 / 누락 8 / 위반 7
  갑의 일방적 해지권: 정상 7 / 누락 7 / 위반 7
  을의 해제권 배제/제한: 정상 7 / 누락 7 / 위반 8
  손해배상 범위 일방적 제한: 정상 7 / 누락 7 / 위반 9


In [ ]:
from google.colab import files
files.download("/content/contract_risk_augmented_v1.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
from google.colab import files
uploaded = files.upload()  # contract_risk_augmented_v2.json 선택

Saving contract_risk_augmented_v2.json to contract_risk_augmented_v2.json


In [3]:
# 형식 변환
import json

SYSTEM_PROMPT = "당신은 공공 SW 계약서의 위험 조항을 탐지하는 전문가입니다. 주어진 계약 조항과 참고 기준을 바탕으로 위험 여부를 판단하고 근거를 제시하십시오."

with open("contract_risk_augmented_v2.json", "r", encoding="utf-8") as f:
    data = json.load(f)

with open("contract_risk_sft_v1.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        # instruction + input 합치기
        user_content = f"{item['instruction']}\n\n{item['input']}"

        converted = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": item["output"]}
            ]
        }
        f.write(json.dumps(converted, ensure_ascii=False) + "\n")

print(f"변환 완료: {len(data)}개")
files.download("contract_risk_sft_v1.jsonl")

변환 완료: 196개
